# Travel-Mode Constraint Function

## Overview

This notebook defines and validates a reusable demographic-informed travel-mode constraint function. The function applies logical feasibility rules based on age and household car availability to the trip-purpose-conditioned travel-mode probability distributions derived from NTS0409a.

For profiles aged 0–4 or 5–11, `Car or van driver` and `Motorcycle` are removed. Where the car-ownership attribute indicates `No cars or vans in household`, `Car or van driver` is removed for all age groups. For profiles aged 0–4 or 5–11 from households without a car or van, `Car or van passenger` is additionally removed. `Taxi or minicab` remains available because it does not depend on household vehicle availability.

After unavailable travel modes are removed, the remaining probabilities are renormalised to sum to one. Profiles that meet none of the constraint conditions retain their original travel-mode distributions.

Profiles from the constrained representative demographic profile space are used as validation cases only. They do not represent population sampling or realised individuals.

## 1. Import Libraries

In [1]:
import json

from copy import deepcopy
from pathlib import Path

import pandas as pd

## 2. Set Up File Paths

In [2]:
project_root = Path.cwd().parents[1]

population_file = (
    project_root
    / "output"
    / "demographic_profiles"
    / "constrained_representative_population.parquet"
)

behavioural_evidence_file = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence.json"
)

## 3. Load Input Data

In [3]:
population = pd.read_parquet(population_file)

with open(
    behavioural_evidence_file,
    "r",
    encoding="utf-8",
) as file:
    behavioural_evidence = json.load(file)

print("Population loaded:", population.shape)
print("Trip purposes loaded:", len(behavioural_evidence))

Population loaded: (2386560, 14)
Trip purposes loaded: 14


## 4. Define Travel-Mode Constraint Function

### Travel-Mode Constraint

The function accepts an age category and one travel-mode probability distribution.

If the age is 0–4 or 5–11, `Car or van driver` is removed and the remaining probabilities are renormalised. Other age groups retain the original distribution.

In [4]:
def apply_travel_mode_constraint(
    age,
    car_ownership,
    travel_mode_probabilities,
):
    constrained = deepcopy(travel_mode_probabilities)

    young_age_groups = {"0-4", "5-11"}
    no_household_car = (
        car_ownership == "No cars or vans in household"
    )

    modes_to_remove = set()

    # Young children cannot drive a car or motorcycle.
    if age in young_age_groups:
        modes_to_remove.update({
            "Car or van driver",
            "Motorcycle",
        })

    # A household without a car cannot support car-driver travel.
    if no_household_car:
        modes_to_remove.add("Car or van driver")

        # For young children, household car availability is also
        # required for car-passenger travel.
        if age in young_age_groups:
            modes_to_remove.add("Car or van passenger")

    # Preserve unaffected distributions exactly as they are.
    if not modes_to_remove:
        return constrained

    for mode in modes_to_remove:
        constrained.pop(mode, None)

    total = sum(constrained.values())

    if total <= 0:
        raise ValueError(
            "No feasible travel-mode probability remains."
        )

    return {
        mode: probability / total
        for mode, probability in constrained.items()
    }

## 5. Select Test Profiles

In [5]:
test_cases = [
    {
        "Age": "0-4",
        "Car Ownership":
            "1 car or van in household",
        "Expected Removed": {
            "Car or van driver",
            "Motorcycle",
        },
    },
    {
        "Age": "5-11",
        "Car Ownership":
            "1 car or van in household",
        "Expected Removed": {
            "Car or van driver",
            "Motorcycle",
        },
    },
    {
        "Age": "0-4",
        "Car Ownership":
            "No cars or vans in household",
        "Expected Removed": {
            "Car or van driver",
            "Car or van passenger",
            "Motorcycle",
        },
    },
    {
        "Age": "5-11",
        "Car Ownership":
            "No cars or vans in household",
        "Expected Removed": {
            "Car or van driver",
            "Car or van passenger",
            "Motorcycle",
        },
    },
    {
        "Age": "23-34",
        "Car Ownership":
            "No cars or vans in household",
        "Expected Removed": {
            "Car or van driver",
        },
    },
    {
        "Age": "23-34",
        "Car Ownership":
            "1 car or van in household",
        "Expected Removed": set(),
    },
]

profile_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

test_profiles = []

for case in test_cases:

    candidates = population[
        (
            population["Age"]
            == case["Age"]
        )
        & (
            population["Car Ownership"]
            == case["Car Ownership"]
        )
    ]

    if candidates.empty:
        raise ValueError(
            "No feasible profile found for "
            f"age={case['Age']} and "
            "car ownership="
            f"{case['Car Ownership']}."
        )

    profile = (
        candidates
        .iloc[0][profile_columns]
        .to_dict()
    )

    profile["Expected Removed"] = (
        case["Expected Removed"]
    )

    test_profiles.append(profile)

test_profiles_df = pd.DataFrame([
    {
        **profile,
        "Expected Removed": (
            ", ".join(sorted(
                profile["Expected Removed"]
            ))
            if profile["Expected Removed"]
            else "None"
        ),
    }
    for profile in test_profiles
])

display(test_profiles_df)

,Age,Gender,Household Type,Economic Activity Status,Industry,Ethnic Group,Distance travelled to work,Car Ownership,Expected Removed
0,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,1 car or van in household,"Car or van driver, Motorcycle"
1,5-11,Female,Couple household: With dependent children,Does not apply,Does not apply,Mixed or Multiple ethnic groups: Other Mixed o...,Does not apply,1 car or van in household,"Car or van driver, Motorcycle"
2,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,No cars or vans in household,"Car or van driver, Car or van passenger, Motor..."
3,5-11,Female,Couple household: With dependent children,Does not apply,Does not apply,Mixed or Multiple ethnic groups: Other Mixed o...,Does not apply,No cars or vans in household,"Car or van driver, Car or van passenger, Motor..."
4,23-34,Female,Couple household: All children non-dependent,Economically active and a full-time student: I...,J Information and communication,"Asian, Asian British or Asian Welsh: Other Asian",10km to less than 30km,No cars or vans in household,Car or van driver
5,23-34,Female,Couple household: All children non-dependent,Economically active and a full-time student: I...,J Information and communication,"Asian, Asian British or Asian Welsh: Other Asian",10km to less than 30km,1 car or van in household,None


## 6. Test and Validate Function

In [6]:
validation_summary = []

for profile in test_profiles:

    age = profile["Age"]

    car_ownership = (
        profile["Car Ownership"]
    )

    expected_removed = set(
        profile["Expected Removed"]
    )

    mode_sets = []

    for purpose, evidence in (
        behavioural_evidence.items()
    ):

        record = (
            evidence[
                "travel_mode_profile"
            ][0]
        )

        original = {
            mode: probability
            for mode, probability
            in record.items()
            if mode != "Purpose"
        }

        constrained = (
            apply_travel_mode_constraint(
                age=age,
                car_ownership=car_ownership,
                travel_mode_probabilities=(
                    original
                ),
            )
        )

        original_modes = set(original)
        remaining_modes = set(constrained)

        removed_modes = (
            original_modes
            - remaining_modes
        )

        # Validate the expected modes
        assert removed_modes == (
            expected_removed
        ), (
            f"Unexpected modes removed for "
            f"age={age}, "
            f"car ownership={car_ownership}, "
            f"purpose={purpose}: "
            f"{removed_modes}"
        )

        # Validate probability sum
        assert abs(
            sum(constrained.values()) - 1
        ) < 1e-8

        # Taxi must always remain available
        assert (
            "Taxi or minicab"
            in constrained
        )

        # Unaffected distributions must
        # remain exactly unchanged
        if not expected_removed:
            assert constrained == original

        mode_sets.append(
            remaining_modes
        )

    # All purposes should retain the
    # same set of permitted modes
    first_mode_set = mode_sets[0]

    assert all(
        modes == first_mode_set
        for modes in mode_sets
    )

    validation_summary.append({
        "Age": age,
        "Car Ownership": car_ownership,
        "Travel Mode Count": len(
            first_mode_set
        ),
        "Removed Travel Mode": (
            ", ".join(sorted(
                expected_removed
            ))
            if expected_removed
            else "None"
        ),
        "Remaining Travel Modes": (
            ", ".join(sorted(
                first_mode_set
            ))
        ),
        "Probability Validation": "PASS",
        "Unchanged Validation": (
            "PASS"
            if not expected_removed
            else "Not applicable"
        ),
    })

validation_summary = pd.DataFrame(
    validation_summary
)

display(validation_summary)

,Age,Car Ownership,Travel Mode Count,Removed Travel Mode,Remaining Travel Modes,Probability Validation,Unchanged Validation
0,0-4,1 car or van in household,9,"Car or van driver, Motorcycle","Car or van passenger, Non-local bus, Other Pri...",PASS,Not applicable
1,5-11,1 car or van in household,9,"Car or van driver, Motorcycle","Car or van passenger, Non-local bus, Other Pri...",PASS,Not applicable
2,0-4,No cars or vans in household,8,"Car or van driver, Car or van passenger, Motor...","Non-local bus, Other Private Transport, Other ...",PASS,Not applicable
3,5-11,No cars or vans in household,8,"Car or van driver, Car or van passenger, Motor...","Non-local bus, Other Private Transport, Other ...",PASS,Not applicable
4,23-34,No cars or vans in household,10,Car or van driver,"Car or van passenger, Motorcycle, Non-local bu...",PASS,Not applicable
5,23-34,1 car or van in household,11,None,"Car or van driver, Car or van passenger, Motor...",PASS,PASS


## Summary

The reusable travel-mode constraint function was successfully validated across six feasible demographic cases representing different combinations of age and household car availability.

For profiles aged 0–4 or 5–11 from households with a car or van, `Car or van driver` and `Motorcycle` were removed, while `Car or van passenger` remained available. For profiles in the same age groups from households without a car or van, `Car or van passenger` was additionally removed.

For the adult profile without a household car or van, only `Car or van driver` was removed. The adult control profile with household car availability retained all eleven original travel modes.

For every adjusted trip-purpose-conditioned distribution, the remaining probabilities were successfully renormalised to sum to one. `Taxi or minicab` remained available in all cases, and the unaffected adult control distributions remained exactly unchanged.